# Lesson 04 — Frames → Embeddings (The GPU Magic)

This is the core lesson. We run **CLIP** on every extracted frame to produce an **embedding** — a vector of 512 numbers that captures the *meaning* of that image.

### What is CLIP?
CLIP (Contrastive Language–Image Pretraining) learns to put images and text in the **same vector space**. A photo of a dog and the text "a dog" end up near each other. A photo of a cat is far away.

### Why GPU?
CLIP's image encoder is a Vision Transformer (ViT). On CPU: ~300 ms/image. On GPU: ~5 ms/image — processing 16 images simultaneously. For 200 frames: **CPU ≈ 60 s vs GPU ≈ 1 s.**

## Step 1 — Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]
print(f"S3 bucket : {S3_BUCKET}")

## Step 2 — Submit the embedding job to Batch

The job will:
1. Download all frames from `s3://<bucket>/frames/sample/`
2. Run CLIP ViT-B/32 on them in batches of 16 (on the GPU)
3. Save `embeddings.npy` and `frame_keys.json` to `s3://<bucket>/embeddings/sample/`

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "submit_job.py", "--video-stem", "sample"],
)
print("Exit code:", result.returncode)

## Step 3 — Download embeddings and inspect them

In [ ]:
import io
import json
import boto3
import numpy as np

s3 = boto3.client("s3")

# Download embeddings
obj        = s3.get_object(Bucket=S3_BUCKET, Key="embeddings/sample/embeddings.npy")
embeddings = np.load(io.BytesIO(obj["Body"].read()))   # shape: (N, 512)

# Download frame key list
obj        = s3.get_object(Bucket=S3_BUCKET, Key="embeddings/sample/frame_keys.json")
frame_keys = json.loads(obj["Body"].read())

print(f"Embeddings shape : {embeddings.shape}")
print(f"  → {embeddings.shape[0]} frames, each represented as {embeddings.shape[1]} numbers")
print(f"First embedding  : {embeddings[0, :5]} ... (first 5 of 512 values)")

## Step 4 — Visualise: project to 2D with PCA

512 dimensions are hard to see. We use PCA to squash them to 2D so we can plot the frames as dots.
Frames that look similar (e.g. consecutive frames of the same scene) cluster together.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca    = PCA(n_components=2)
coords = pca.fit_transform(embeddings)   # shape: (N, 2)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=range(len(coords)), cmap="viridis", s=30, alpha=0.8)
plt.colorbar(label="Frame index (time →)")
plt.title("CLIP embeddings projected to 2D (PCA)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.tight_layout()
plt.show()

print(f"Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}")

## Key Takeaway

> CLIP turns every frame into a 512-number fingerprint. Frames that look similar → similar fingerprints → close in space. Text that *describes* a frame → also close in space. **That's the magic we'll use in Lesson 05 to search by text.**

---

## Next lesson → [05 — Vector Search](../05-vector-search/notebook.ipynb)

We'll type a text query and find the matching frame — no SQL, no tags, just math on vectors.